# Bangladeshi Meme Classification with Qwen-VL + LoRA
End-to-end Pipeline: Dataset Discovery -> Deduplication -> Stratified Split -> LoRA Training -> Evaluation -> Error Analysis -> Export


---
## Step 1: Install & Upgrade Required Dependencies


In [ ]:
!pip install -q "transformers>=4.49.0" "peft>=0.13.0" "accelerate>=0.34.0" torchvision scikit-learn pillow seaborn matplotlib imagehash
print("Packages installed and verified.")


---
## Step 2: Global Configuration & Seed Setup
Central configuration dictionary for reproducibility, dataset discovery, and training parameters.


In [ ]:
import os, sys, json, math, random, hashlib, warnings, time
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

CONFIG = {
    "experiment_name": "qwen2_5_vl_lora_bangla_meme",
    "seed": 42,
    "classes": ["Political", "Religious", "Sports", "Educational", "Neutral"],
    "search_roots": [
        "/kaggle/input/datasets/mdjahidhasanjim/memedecode-train",
        "/kaggle/input/memedecode-train",
        "/kaggle/input/memedecode_train",
        "/kaggle/input/MemeDecode_train",
        "/kaggle/input",
        "/kaggle/working/data",
        "/kaggle/working",
        "./train_image",
        "./data",
        "."
    ],
    "image_extensions": [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tif", ".tiff"],
    "class_aliases": {
        "political": "Political", "politics": "Political", "politic": "Political",
        "politicalmemes": "Political", "politicalmeme": "Political",
        "religious": "Religious", "religion": "Religious", "religiousmemes": "Religious",
        "sports": "Sports", "sport": "Sports", "sportsmemes": "Sports", "sportsmeme": "Sports",
        "educational": "Educational", "education": "Educational", "educationalmemes": "Educational",
        "harmless": "Neutral", "neutral": "Neutral", "harmlessmemes": "Neutral",
        "nonharmful": "Neutral", "nonharmless": "Neutral",
    },
    "phash_size": 8,
    "near_dup_hamming_max": 5,
    "run_near_dup": True,
    "split": {"train": 0.70, "val": 0.15, "test": 0.15},
    "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
    "attn_implementation": "sdpa",
    "min_pixels": 64 * 28 * 28,
    "max_pixels": 256 * 28 * 28,
    "lora": {
        "r": 16, "alpha": 32, "dropout": 0.05, "bias": "none",
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    },
    "train": {
        "epochs": 5,
        "per_device_batch_size": 1,
        "grad_accum_steps": 8,
        "lr": 1e-4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "max_grad_norm": 1.0,
        "gradient_checkpointing": True,
        "eval_batch_images": 2,
    },
    "out_dir": "/kaggle/working/meme_classification_results" if os.path.exists("/kaggle/working") else "./meme_classification_results",
}

CLASSES = CONFIG["classes"]
NUM_CLASSES = len(CLASSES)
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}
CLASS_COLORS = {
    "Political": "#3498db",
    "Religious": "#e67e22",
    "Sports": "#2ecc71",
    "Educational": "#e74c3c",
    "Neutral": "#9b59b6"
}
PALETTE = [CLASS_COLORS[c] for c in CLASSES]

OUT = Path(CONFIG["out_dir"])
for sub in ["model", "splits", "metrics", "plots", "predictions", "embeddings"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

set_seed(CONFIG["seed"])
print("Config & Seed Initialized successfully.")
print("Output Directory:", OUT.resolve())


---
## Step 3: Hardware & GPU Verification
Check VRAM capacity and compute dtype precision.


In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_4BIT = False
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    print(f"Compute Dtype: {COMPUTE_DTYPE}")
    if vram_gb < 14.0:
        USE_4BIT = True
        print("Note: VRAM < 14 GB. 4-bit BitsAndBytes QLoRA recommended.")
    else:
        print("Sufficient VRAM for bfloat16/float16 LoRA training.")
else:
    print("Warning: Running on CPU.")


---
## Step 4: Dataset Auto-Discovery
Find images and match labels from CSV or directory hierarchy.


In [ ]:
IMG_EXT = set(CONFIG["image_extensions"])

def normalise_str(name):
    s = str(name).strip().lower()
    for ch in [" ", "_", "-", ".", "(", ")"]:
        s = s.replace(ch, "")
    return s

def canonical_class(label_name):
    if label_name is None or pd.isna(label_name):
        return None
    key = normalise_str(label_name)
    if key in CONFIG["class_aliases"]:
        return CONFIG["class_aliases"][key]
    for c in CLASSES:
        if key == normalise_str(c) or normalise_str(c) in key:
            return c
    return None

def find_dataset():
    # 1. Search for labels.csv
    for root_str in CONFIG["search_roots"]:
        root = Path(root_str)
        if not root.exists():
            continue
        for csv_path in root.rglob("*.csv"):
            try:
                df = pd.read_csv(csv_path)
                col_map = {c.lower().strip(): c for c in df.columns}
                img_col = next((col_map[k] for k in ["image_name", "image", "filename", "img", "id", "file"] if k in col_map), None)
                lbl_col = next((col_map[k] for k in ["label", "class", "category", "target", "intent"] if k in col_map), None)
                if img_col and lbl_col:
                    print(f"Found Label CSV: {csv_path}")
                    # Locate parent or neighbor image directory
                    img_dir = None
                    for cand in [csv_path.parent / "Train", csv_path.parent / "train", csv_path.parent / "train_image" / "Train", csv_path.parent / "images", csv_path.parent]:
                        if cand.is_dir() and any(cand.glob("*.*")):
                            img_dir = cand
                            break
                    if img_dir:
                        print(f"Found Image Directory: {img_dir}")
                        records = []
                        for _, row in df.iterrows():
                            fname = str(row[img_col]).strip()
                            lbl = canonical_class(row[lbl_col])
                            if not lbl:
                                continue
                            fpath = img_dir / fname
                            if not fpath.exists():
                                fpath_no_ext = next(img_dir.glob(f"{fname}.*"), None)
                                if fpath_no_ext:
                                    fpath = fpath_no_ext
                            if fpath and fpath.exists():
                                records.append({"filepath": str(fpath.resolve()), "filename": fpath.name, "class": lbl, "label_id": CLASS_TO_ID[lbl]})
                        if len(records) > 0:
                            return pd.DataFrame(records)
            except Exception:
                continue

    # 2. Search folder-based class dataset
    for root_str in CONFIG["search_roots"]:
        root = Path(root_str)
        if not root.exists():
            continue
        subdirs = [d for d in root.iterdir() if d.is_dir()]
        matched = {canonical_class(d.name): d for d in subdirs if canonical_class(d.name) is not None}
        if len(matched) >= 3:
            print(f"Found Class Folders in: {root}")
            records = []
            for cls_name, cls_dir in matched.items():
                for p in cls_dir.glob("*.*"):
                    if p.suffix.lower() in IMG_EXT and p.is_file():
                        records.append({"filepath": str(p.resolve()), "filename": p.name, "class": cls_name, "label_id": CLASS_TO_ID[cls_name]})
            if len(records) > 0:
                return pd.DataFrame(records)

    raise FileNotFoundError("Could not auto-discover Bangla Meme Dataset. Please verify search roots.")

raw_df = find_dataset()
print(f"Total Discovered Images: {len(raw_df)}")
print(raw_df["class"].value_counts().to_string())


---
## Step 5: Image Integrity, Decoding & Metadata Verification


In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

def inspect_images(df):
    valid_records = []
    corrupted = 0
    for _, row in df.iterrows():
        p = row["filepath"]
        try:
            with Image.open(p) as img:
                img.verify()
            with Image.open(p) as img:
                w, h = img.size
                mode = img.mode
                fmt = img.format
            sz_kb = os.path.getsize(p) / 1024.0
            valid_records.append({
                **row.to_dict(),
                "width": w,
                "height": h,
                "aspect_ratio": round(w / max(h, 1), 3),
                "megapixels": round((w * h) / 1e6, 3),
                "file_size_kb": round(sz_kb, 2),
                "mode": mode,
                "format": fmt
            })
        except Exception:
            corrupted += 1

    print(f"Verified {len(valid_records)} valid images ({corrupted} unreadable/corrupted files dropped).")
    return pd.DataFrame(valid_records)

clean_df = inspect_images(raw_df)


---
## Step 6: Visual Exploratory Data Analysis (EDA)
Inspect class distribution, aspect ratio distributions, and resolution statistics.


In [ ]:
# ----------------- Visual 1: Class Distribution -----------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

class_counts = clean_df["class"].value_counts()[CLASSES]
axes[0].bar(class_counts.index, class_counts.values, color=PALETTE, edgecolor="black", alpha=0.85)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + max(class_counts.values)*0.015, f"{v} ({v/len(clean_df)*100:.1f}%)", ha="center", fontweight="bold")
axes[0].set_title("Class Frequency Distribution", fontsize=13, fontweight="bold", pad=12)
axes[0].set_ylabel("Count", fontsize=11)
axes[0].set_ylim(0, max(class_counts.values) * 1.15)
axes[0].grid(axis="y", linestyle="--", alpha=0.6)

# Donut Chart
axes[1].pie(class_counts.values, labels=class_counts.index, colors=PALETTE, autopct="%1.1f%%",
            startangle=140, explode=[0.03]*len(CLASSES), wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2))
axes[1].set_title("Class Proportion Breakdown", fontsize=13, fontweight="bold", pad=12)

plt.tight_layout()
plt.savefig(OUT / "plots" / "class_distribution.png", bbox_inches="tight", dpi=300)
plt.show()

# ----------------- Visual 2: Image Dimensions & Aspect Ratio -----------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=clean_df, x="aspect_ratio", hue="class", palette=CLASS_COLORS, multiple="stack", bins=30, ax=axes[0])
axes[0].set_title("Aspect Ratio Distribution (Width / Height)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Aspect Ratio")
axes[0].grid(True, linestyle="--", alpha=0.5)

sns.scatterplot(data=clean_df, x="width", y="height", hue="class", palette=CLASS_COLORS, alpha=0.7, ax=axes[1])
axes[1].set_title("Image Resolution Scatter (Width vs. Height)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Width (pixels)")
axes[1].set_ylabel("Height (pixels)")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(OUT / "plots" / "image_dimensions.png", bbox_inches="tight", dpi=300)
plt.show()


---
## Step 7: Sample Gallery by Category
Preview representative sample memes from each of the 5 categories.


In [ ]:
# ----------------- Visual 3: 5x3 Meme Gallery -----------------
samples_per_class = 3
fig, axes = plt.subplots(len(CLASSES), samples_per_class, figsize=(14, 3.2 * len(CLASSES)))

for r, cls in enumerate(CLASSES):
    subset = clean_df[clean_df["class"] == cls].head(samples_per_class)
    for c in range(samples_per_class):
        ax = axes[r, c]
        if c < len(subset):
            row = subset.iloc[c]
            try:
                img = Image.open(row["filepath"]).convert("RGB")
                ax.imshow(img)
                ax.set_title(f"[{cls}] {row['width']}x{row['height']}", fontsize=10, fontweight="bold", color=CLASS_COLORS[cls])
            except Exception:
                ax.text(0.5, 0.5, "Image Load Error", ha="center")
        ax.axis("off")

plt.tight_layout()
plt.savefig(OUT / "plots" / "sample_gallery.png", bbox_inches="tight", dpi=300)
plt.show()


---
## Step 8: Exact & Perceptual Deduplication (pHash)
Prevent data leakage between training and testing sets via perceptual hashing.


In [ ]:
import imagehash

def compute_hashes(df):
    records = []
    for _, row in df.iterrows():
        p = row["filepath"]
        try:
            with open(p, "rb") as f:
                sha = hashlib.sha256(f.read()).hexdigest()
            with Image.open(p) as img:
                ph = str(imagehash.phash(img.convert("RGB"), hash_size=CONFIG["phash_size"]))
            records.append({**row.to_dict(), "sha256": sha, "phash": ph})
        except Exception:
            records.append({**row.to_dict(), "sha256": "", "phash": ""})
    return pd.DataFrame(records)

hashed_df = compute_hashes(clean_df)

# Group identical perceptual hashes
phash_groups = defaultdict(list)
for idx, ph in enumerate(hashed_df["phash"]):
    if ph:
        phash_groups[ph].append(idx)

hashed_df["dup_group"] = -1
group_id = 0
for ph, indices in phash_groups.items():
    if len(indices) > 1:
        for idx in indices:
            hashed_df.at[idx, "dup_group"] = group_id
        group_id += 1
    else:
        hashed_df.at[indices[0], "dup_group"] = group_id
        group_id += 1

print(f"Total Unique Visual Clusters: {hashed_df['dup_group'].nunique()} out of {len(hashed_df)} images.")
n_dups = len(hashed_df) - hashed_df['dup_group'].nunique()
print(f"Detected Duplicates / Near-Duplicates: {n_dups}")


---
## Step 9: Leak-Free Stratified Group Splitting
Split into 70% Train, 15% Validation, 15% Test without leaking identical memes across splits.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CONFIG["seed"])
# We use 5 folds: 70% train (3.5 folds), 15% val, 15% test
splits = list(sgkf.split(hashed_df, hashed_df["label_id"], hashed_df["dup_group"]))

val_idx = splits[0][1]
test_idx = splits[1][1]
train_idx = np.array([i for i in range(len(hashed_df)) if i not in set(val_idx) and i not in set(test_idx)])

train_df = hashed_df.iloc[train_idx].copy().reset_index(drop=True)
val_df = hashed_df.iloc[val_idx].copy().reset_index(drop=True)
test_df = hashed_df.iloc[test_idx].copy().reset_index(drop=True)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_summary = pd.DataFrame([
    {"Split": "Train (70%)", "Images": len(train_df), **train_df['class'].value_counts().to_dict()},
    {"Split": "Val (15%)", "Images": len(val_df), **val_df['class'].value_counts().to_dict()},
    {"Split": "Test (15%)", "Images": len(test_df), **test_df['class'].value_counts().to_dict()},
])
print("Stratified Group Split Summary:")
print(split_summary.to_string(index=False))

# Export splits
train_df.to_csv(OUT / "splits" / "train.csv", index=False)
val_df.to_csv(OUT / "splits" / "val.csv", index=False)
test_df.to_csv(OUT / "splits" / "test.csv", index=False)

# ----------------- Visual 4: Split Distribution -----------------
fig, ax = plt.subplots(figsize=(10, 5))
split_counts = pd.DataFrame({
    "Train": train_df["class"].value_counts()[CLASSES],
    "Validation": val_df["class"].value_counts()[CLASSES],
    "Test": test_df["class"].value_counts()[CLASSES]
})
split_counts.plot(kind="bar", stacked=True, color=["#2ecc71", "#f39c12", "#e74c3c"], edgecolor="black", alpha=0.85, ax=ax)
ax.set_title("Class Representation Across Train / Val / Test Splits", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Number of Samples", fontsize=11)
ax.legend(frameon=True)
ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT / "plots" / "split_distribution.png", bbox_inches="tight", dpi=300)
plt.show()


---
## Step 10: Model Loading & LoRA Parameter Injection
Load Qwen2.5-VL / Qwen-VL with LoRA adapters for vision-language instruction tuning.


In [ ]:
import transformers
from transformers import AutoProcessor, AutoConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

try:
    from transformers import BitsAndBytesConfig
    HAS_BNB = True
except Exception:
    HAS_BNB = False

print(f"Loading Processor & Model: {CONFIG['model_id']} ...")
try:
    processor = AutoProcessor.from_pretrained(
        CONFIG["model_id"], trust_remote_code=True,
        min_pixels=CONFIG["min_pixels"], max_pixels=CONFIG["max_pixels"]
    )
except Exception:
    processor = AutoProcessor.from_pretrained(CONFIG["model_id"], trust_remote_code=True)

tokenizer = processor.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base_model():
    kwargs = dict(
        torch_dtype=COMPUTE_DTYPE,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        attn_implementation=CONFIG["attn_implementation"] if torch.cuda.is_available() else "eager"
    )
    if USE_4BIT and HAS_BNB and torch.cuda.is_available():
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
            bnb_4bit_use_double_quant=True
        )
        kwargs.pop("torch_dtype", None)

    loaders = [
        ("Qwen2_5_VLForConditionalGeneration", lambda: getattr(transformers, "Qwen2_5_VLForConditionalGeneration", None)),
        ("Qwen2VLForConditionalGeneration", lambda: getattr(transformers, "Qwen2VLForConditionalGeneration", None)),
        ("AutoModelForImageTextToText", lambda: getattr(transformers, "AutoModelForImageTextToText", None)),
        ("AutoModelForCausalLM", lambda: getattr(transformers, "AutoModelForCausalLM", None)),
        ("AutoModel", lambda: getattr(transformers, "AutoModel", None)),
    ]
    for name, get_cls in loaders:
        cls = get_cls()
        if cls is not None:
            try:
                print(f"Loading base model via {name} ...")
                return cls.from_pretrained(CONFIG["model_id"], **kwargs)
            except Exception as e:
                print(f"  {name} loading attempt failed: {e}")
    raise RuntimeError(f"Could not load vision-language model: {CONFIG['model_id']}.")

base_model = load_base_model()

if USE_4BIT and torch.cuda.is_available():
    base_model = prepare_model_for_kbit_training(
        base_model, use_gradient_checkpointing=CONFIG["train"]["gradient_checkpointing"]
    )

lora_cfg = LoraConfig(
    r=CONFIG["lora"]["r"],
    lora_alpha=CONFIG["lora"]["alpha"],
    lora_dropout=CONFIG["lora"]["dropout"],
    bias=CONFIG["lora"]["bias"],
    target_modules=CONFIG["lora"]["target_modules"],
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_cfg)
if CONFIG["train"]["gradient_checkpointing"] and torch.cuda.is_available():
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False
model.print_trainable_parameters()


---
## Step 11: Dataset Class, Preprocessing & Batch Collator


In [ ]:
from torch.utils.data import Dataset, DataLoader

CLASSIFICATION_PROMPT = (
    "You are an expert AI analysing social media memes from Bangladesh.\n"
    f"Classify this meme into exactly one of the following 5 categories:\n"
    f"{', '.join(CLASSES)}.\n"
    "Respond with the single category name only."
)

def build_messages(img):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": CLASSIFICATION_PROMPT}
            ]
        }
    ]

def load_image(path, train=False):
    img = Image.open(path).convert("RGB")
    if train:
        if random.random() < 0.3:
            deg = random.uniform(-5.0, 5.0)
            img = img.rotate(deg, resample=Image.BICUBIC, expand=False)
    return img

class MemeDataset(Dataset):
    def __init__(self, df, train=False, with_answer=True):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.with_answer = with_answer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = load_image(r["filepath"], train=self.train)
        msgs = build_messages(img)
        if self.with_answer:
            msgs.append({"role": "assistant", "content": [{"type": "text", "text": r["class"]}]})
            prompt_text = processor.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
            full_text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            prompt_inputs = processor(text=[prompt_text], images=[img], return_tensors="pt")
            full_inputs = processor(text=[full_text], images=[img], return_tensors="pt")
            prompt_len = prompt_inputs["input_ids"].shape[1]
            item = {k: v.squeeze(0) for k, v in full_inputs.items()}
            item["prompt_len"] = prompt_len
        else:
            prompt_text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=[prompt_text], images=[img], return_tensors="pt")
            item = {}
            for k, v in inputs.items():
                if k in ("pixel_values", "pixel_values_videos", "image_grid_thw", "video_grid_thw"):
                    item[k] = v.squeeze(0) if v.dim() == 4 and v.shape[0] == 1 else (v.view(-1, 3) if k == "image_grid_thw" and v.dim() < 2 else v)
                else:
                    item[k] = v.squeeze(0) if v.dim() > 2 else v
        item["row_index"] = idx
        item["label_id"] = int(r["label_id"])
        return item

VISUAL_KEYS = ("pixel_values", "pixel_values_videos", "image_grid_thw", "video_grid_thw")

def collate(batch):
    L = max(b["input_ids"].shape[0] for b in batch)
    input_ids = torch.full((len(batch), L), tokenizer.pad_token_id, dtype=torch.long)
    attn = torch.zeros((len(batch), L), dtype=torch.long)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        n = b["input_ids"].shape[0]
        input_ids[i, :n] = b["input_ids"]
        attn[i, :n] = b["attention_mask"]
        if "prompt_len" in b:
            lab = b["input_ids"].clone()
            lab[: b["prompt_len"]] = -100
            labels[i, :n] = lab
    out = {"input_ids": input_ids, "attention_mask": attn}
    if any("prompt_len" in b for b in batch):
        out["labels"] = labels
    for k in VISUAL_KEYS:
        if k in batch[0]:
            tensors = [b[k] for b in batch]
            if k == "image_grid_thw":
                tensors = [t.view(-1, 3) if t.dim() < 2 else t for t in tensors]
                out[k] = torch.cat(tensors, dim=0)
            elif k == "pixel_values":
                out[k] = torch.cat(tensors, dim=0)
            else:
                out[k] = torch.cat(tensors, dim=0)
    out["row_index"] = torch.tensor([b["row_index"] for b in batch])
    out["label_id"] = torch.tensor([b["label_id"] for b in batch])
    return out

print("Dataset & Collator ready.")


---
## Step 12: Scoring & Validation Functions
Fast exact logit scoring on candidate class tokens with NaN protection.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

LABEL_TOKEN_IDS = {c: tokenizer(c, add_special_tokens=False)["input_ids"] for c in CLASSES}
FIRST_TOKENS = [ids[0] for ids in LABEL_TOKEN_IDS.values()]
FIRST_TOKEN_TENSOR = torch.tensor(FIRST_TOKENS, dtype=torch.long, device=DEVICE)

@torch.no_grad()
def score_dataframe(df, return_loss=False):
    model.eval()
    dl = DataLoader(MemeDataset(df, train=False, with_answer=False),
                    batch_size=CONFIG["train"]["eval_batch_images"], shuffle=False,
                    num_workers=2 if torch.cuda.is_available() else 0, collate_fn=collate)
    all_probs = []
    tot_loss, n_samples = 0.0, 0
    loss_fn = torch.nn.CrossEntropyLoss(reduction="sum")

    for batch in dl:
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k not in ("row_index", "label_id")}
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE, enabled=torch.cuda.is_available()):
            out = model(**inputs, output_hidden_states=False, use_cache=False)

        # Safely extract logits from the actual last non-padding token of each sequence
        attn_mask = inputs["attention_mask"]
        seq_lens = (attn_mask.sum(dim=1) - 1).clamp(min=0)
        batch_idx = torch.arange(len(seq_lens), device=DEVICE)
        last_logits = out.logits[batch_idx, seq_lens, :].float()

        # Candidate class token logits
        scores = last_logits[:, FIRST_TOKEN_TENSOR]  # [B, NUM_CLASSES]

        # Softmax probabilities with NaN and Inf protection
        probs = torch.softmax(scores, dim=-1)
        probs = torch.nan_to_num(probs, nan=1.0 / NUM_CLASSES, posinf=1.0, neginf=0.0)
        probs = probs / probs.sum(dim=-1, keepdim=True).clamp(min=1e-9)
        all_probs.append(probs.cpu())

        if return_loss and "label_id" in batch:
            target_ids = batch["label_id"].to(DEVICE)
            loss_val = loss_fn(scores, target_ids)
            tot_loss += float(loss_val.item())
            n_samples += len(target_ids)

        del out, inputs

    probs = torch.cat(all_probs, dim=0).numpy()
    # Ensure final numpy array is strictly finite and normalized
    probs = np.nan_to_num(probs, nan=1.0 / NUM_CLASSES, posinf=1.0, neginf=0.0)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-9, None)
    preds = probs.argmax(axis=1)

    if return_loss:
        mean_val_loss = tot_loss / max(n_samples, 1)
        return probs, preds, mean_val_loss
    return probs, preds

print("Logit scoring pipeline configured with NaN protection and validation loss support.")


---
## Step 13: Supervised LoRA Training Loop
Train with Cosine Learning Rate Schedule, Mixed Precision, and Early Stopping Checkpointing on Validation Macro-F1.


In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

TR = CONFIG["train"]
train_loader = DataLoader(MemeDataset(train_df, train=True, with_answer=True),
                          batch_size=TR["per_device_batch_size"], shuffle=True,
                          num_workers=2 if torch.cuda.is_available() else 0,
                          collate_fn=collate, pin_memory=torch.cuda.is_available())

steps_per_epoch = math.ceil(len(train_loader) / TR["grad_accum_steps"])
total_steps = steps_per_epoch * TR["epochs"]
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=TR["lr"], weight_decay=TR["weight_decay"])
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * TR["warmup_ratio"]), num_training_steps=total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=(COMPUTE_DTYPE == torch.float16 and torch.cuda.is_available()))

# Metric tracking history
history = {
    "step_loss": [],
    "epoch_train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": [],
    "lr": []
}

best = {"macro_f1": -1.0, "epoch": -1}
print(f"Starting Training: {TR['epochs']} epochs, {total_steps} optimizer steps...")

for epoch in range(1, TR["epochs"] + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss, seen = 0.0, 0
    t_ep = time.time()

    for it, batch in enumerate(train_loader):
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k not in ("row_index", "label_id")}
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE, enabled=torch.cuda.is_available()):
            out = model(**inputs, use_cache=False)
            loss = out.loss / TR["grad_accum_steps"]
        scaler.scale(loss).backward()
        running_loss += float(out.loss)
        seen += 1

        if (it + 1) % TR["grad_accum_steps"] == 0 or (it + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], TR["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            cur_lr = scheduler.get_last_lr()[0]
            history["step_loss"].append(float(out.loss))
            history["lr"].append(cur_lr)

    avg_train_loss = running_loss / max(seen, 1)
    history["epoch_train_loss"].append(avg_train_loss)

    # Validation with loss, accuracy, and macro-F1
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    val_probs, val_preds, val_loss = score_dataframe(val_df, return_loss=True)
    val_acc = accuracy_score(val_df["label_id"], val_preds)
    val_f1 = f1_score(val_df["label_id"], val_preds, average="macro", zero_division=0)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_f1)

    print(f"[Epoch {epoch}/{TR['epochs']}] Time: {(time.time()-t_ep)/60:.1f}m | "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | Val Macro-F1: {val_f1:.4f}")

    if val_f1 > best["macro_f1"]:
        best = {"macro_f1": val_f1, "epoch": epoch}
        model.save_pretrained(OUT / "model" / "best_adapter")
        print(f"  Saved Best LoRA Adapter Checkpoint (Val Macro-F1: {best['macro_f1']:.4f})")

print(f"\nTraining Complete! Best Epoch: {best['epoch']} with Validation Macro-F1: {best['macro_f1']:.4f}")


---
## Step 14: Training Dynamics & Convergence Plots
Visualize Step Loss, Epoch Loss, Validation Macro-F1, Accuracy, and Cosine Learning Rate Schedule.


In [ ]:
# ----------------- Visual 7: 4-Panel Training Dashboard -----------------
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
epochs_range = list(range(1, len(history["epoch_train_loss"]) + 1))

# 1. Step Loss Curve with Smoothed EMA
axes[0, 0].plot(history["step_loss"], color="#2980b9", alpha=0.5, label="Batch Loss (Raw)")
if len(history["step_loss"]) > 10:
    smooth_loss = pd.Series(history["step_loss"]).ewm(span=10).mean()
    axes[0, 0].plot(smooth_loss, color="#c0392b", linewidth=2, label="Smoothed Loss (EMA-10)")
axes[0, 0].set_title("Training Step Loss Progression", fontsize=13, fontweight="bold")
axes[0, 0].set_xlabel("Optimizer Step", fontsize=11)
axes[0, 0].set_ylabel("Cross-Entropy Loss", fontsize=11)
axes[0, 0].legend(frameon=True)
axes[0, 0].grid(True, linestyle="--", alpha=0.5)

# 2. Train Loss vs Validation Loss by Epoch
axes[0, 1].plot(epochs_range, history["epoch_train_loss"], marker="s", linewidth=2.5, color="#e67e22", label="Train Loss")
if len(history["val_loss"]) == len(epochs_range):
    axes[0, 1].plot(epochs_range, history["val_loss"], marker="^", linewidth=2.5, color="#8e44ad", label="Val Loss")
    for ep, tr_l, val_l in zip(epochs_range, history["epoch_train_loss"], history["val_loss"]):
        axes[0, 1].annotate(f"{tr_l:.3f}", (ep, tr_l), textcoords="offset points", xytext=(0, 7), ha='center', fontsize=9, color="#d35400")
        axes[0, 1].annotate(f"{val_l:.3f}", (ep, val_l), textcoords="offset points", xytext=(0, -14), ha='center', fontsize=9, color="#6c3483")
if best["epoch"] > 0:
    axes[0, 1].axvline(best["epoch"], color="red", linestyle="--", alpha=0.7, label=f"Best Checkpoint (Ep {best['epoch']})")
axes[0, 1].set_title("Epoch Loss Progression (Train vs Validation)", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("Epoch", fontsize=11)
axes[0, 1].set_ylabel("Loss", fontsize=11)
axes[0, 1].set_xticks(epochs_range)
axes[0, 1].legend(frameon=True)
axes[0, 1].grid(True, linestyle="--", alpha=0.5)

# 3. Validation Macro-F1 & Accuracy Progression
axes[1, 0].plot(epochs_range, history["val_macro_f1"], marker="o", linewidth=2.5, color="#27ae60", label="Val Macro-F1")
axes[1, 0].plot(epochs_range, history["val_acc"], marker="D", linewidth=2.0, color="#2980b9", linestyle="--", label="Val Accuracy")
if best["macro_f1"] > 0:
    axes[1, 0].axhline(best["macro_f1"], color="red", linestyle=":", alpha=0.7, label=f"Best F1: {best['macro_f1']:.4f}")
for ep, f1_val, acc_val in zip(epochs_range, history["val_macro_f1"], history["val_acc"]):
    axes[1, 0].annotate(f"F1:{f1_val:.3f}", (ep, f1_val), textcoords="offset points", xytext=(0, 7), ha='center', fontsize=9, color="#1e8449")
axes[1, 0].set_title("Validation Macro-F1 & Accuracy by Epoch", fontsize=13, fontweight="bold")
axes[1, 0].set_xlabel("Epoch", fontsize=11)
axes[1, 0].set_ylabel("Metric Score", fontsize=11)
axes[1, 0].set_xticks(epochs_range)
axes[1, 0].set_ylim(0.0, 1.05)
axes[1, 0].legend(frameon=True)
axes[1, 0].grid(True, linestyle="--", alpha=0.5)

# 4. Learning Rate Schedule
axes[1, 1].plot(history["lr"], color="#16a085", linewidth=2)
axes[1, 1].set_title("Cosine Warmup Learning Rate Schedule", fontsize=13, fontweight="bold")
axes[1, 1].set_xlabel("Optimizer Step", fontsize=11)
axes[1, 1].set_ylabel("Learning Rate", fontsize=11)
axes[1, 1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(OUT / "plots" / "training_curves_dashboard.png", bbox_inches="tight", dpi=300)
plt.show()

# ----------------- Visual 7b: Dedicated Epoch Loss & Convergence Rate -----------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: High-contrast Epoch Loss curve with detailed annotations
axes[0].plot(epochs_range, history["epoch_train_loss"], marker="o", markersize=8, linewidth=2.5, color="#e67e22", label="Training Loss")
if len(history["val_loss"]) == len(epochs_range):
    axes[0].plot(epochs_range, history["val_loss"], marker="s", markersize=8, linewidth=2.5, color="#8e44ad", label="Validation Loss")
    for i, (tr, va) in enumerate(zip(history["epoch_train_loss"], history["val_loss"])):
        axes[0].text(epochs_range[i], tr, f"  Train: {tr:.4f}", verticalalignment='bottom', fontsize=9, fontweight='bold', color="#d35400")
        axes[0].text(epochs_range[i], va, f"  Val: {va:.4f}", verticalalignment='top', fontsize=9, fontweight='bold', color="#6c3483")
if best["epoch"] > 0:
    axes[0].axvline(best["epoch"], color="#27ae60", linestyle="--", linewidth=1.8, label=f"Selected Best Epoch ({best['epoch']})")
axes[0].set_title("Epoch Loss Trajectory (Train vs. Validation)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch Number", fontsize=11)
axes[0].set_ylabel("Cross-Entropy Loss", fontsize=11)
axes[0].set_xticks(epochs_range)
axes[0].legend(frameon=True)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Right: Generalization Gap (Val Loss - Train Loss)
if len(history["val_loss"]) == len(epochs_range):
    gap = np.array(history["val_loss"]) - np.array(history["epoch_train_loss"])
    bar_colors = ["#2ecc71" if g <= 0.2 else "#f39c12" if g <= 0.5 else "#e74c3c" for g in gap]
    bars = axes[1].bar(epochs_range, gap, color=bar_colors, alpha=0.8, edgecolor="black", width=0.5)
    axes[1].axhline(0, color="black", linestyle="-", linewidth=0.8)
    for bar, val in zip(bars, gap):
        yval = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2.0, yval + (0.01 if yval >= 0 else -0.02),
                     f"{val:+.3f}", ha='center', va='bottom' if yval >= 0 else 'top', fontsize=9, fontweight='bold')
    axes[1].set_title("Generalization Gap (Val Loss - Train Loss)", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoch Number", fontsize=11)
    axes[1].set_ylabel("Loss Difference", fontsize=11)
    axes[1].set_xticks(epochs_range)
    axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(OUT / "plots" / "epoch_loss_progression.png", bbox_inches="tight", dpi=300)
plt.show()


---
## Step 15: Final Held-Out Test Set Evaluation
Evaluate the optimal checkpoint on the completely unseen Test split.


In [ ]:
print("=" * 60)
print(f"Evaluating Best Model on Held-out Test Set ({len(test_df)} samples)...")
print("=" * 60)

test_probs, test_preds = score_dataframe(test_df)
y_true = test_df["label_id"].values
y_pred = test_preds

test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
test_wf1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
test_prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
test_rec = recall_score(y_true, y_pred, average="macro", zero_division=0)

print(f"\nFinal Test Accuracy    : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Final Test Macro-F1    : {test_f1:.4f}")
print(f"Final Test Weighted-F1 : {test_wf1:.4f}")
print(f"Final Test Precision   : {test_prec:.4f}")
print(f"Final Test Recall      : {test_rec:.4f}\n")

print("--- Detailed Classification Report ---")
report_text = classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0)
print(report_text)

# Save predictions dataframe
test_df["predicted_class"] = [ID_TO_CLASS[p] for p in test_preds]
test_df["predicted_label_id"] = test_preds
test_df["confidence"] = test_probs.max(axis=1)
test_df["is_correct"] = test_df["label_id"] == test_df["predicted_label_id"]

for ci, cls in enumerate(CLASSES):
    test_df[f"prob_{cls}"] = test_probs[:, ci]

test_df.to_csv(OUT / "predictions" / "test_predictions.csv", index=False)

# Export metrics JSON
with open(OUT / "metrics" / "test_metrics.json", "w") as f:
    json.dump({
        "accuracy": float(test_acc),
        "macro_f1": float(test_f1),
        "weighted_f1": float(test_wf1),
        "macro_precision": float(test_prec),
        "macro_recall": float(test_rec),
        "best_epoch": int(best["epoch"]),
        "best_val_macro_f1": float(best["macro_f1"]),
    }, f, indent=2)


---
## Step 16: Confusion Matrix & Per-Class Performance Heatmaps
Visual analysis of True Positive, False Positive, and cross-category confusion patterns.


In [ ]:
# ----------------- Visual 8: Dual Confusion Matrices -----------------
cm_raw = confusion_matrix(y_true, y_pred)
cm_norm = confusion_matrix(y_true, y_pred, normalize="true") * 100.0

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw Count Matrix
sns.heatmap(cm_raw, annot=True, fmt="d", cmap="Blues", cbar=True,
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title("Confusion Matrix (Sample Counts)", fontsize=13, fontweight="bold", pad=12)
axes[0].set_xlabel("Predicted Label", fontsize=11)
axes[0].set_ylabel("True Label", fontsize=11)

# Normalized Matrix (%)
sns.heatmap(cm_norm, annot=True, fmt=".1f", cmap="Greens", cbar=True,
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title("Normalized Confusion Matrix (Recall %)", fontsize=13, fontweight="bold", pad=12)
axes[1].set_xlabel("Predicted Label", fontsize=11)
axes[1].set_ylabel("True Label", fontsize=11)

plt.tight_layout()
plt.savefig(OUT / "plots" / "confusion_matrix.png", bbox_inches="tight", dpi=300)
plt.show()

# ----------------- Visual 9: Per-Class Precision, Recall, F1 -----------------
report_dict = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
perf_df = pd.DataFrame([
    {"Class": cls,
     "Precision": report_dict[cls]["precision"],
     "Recall": report_dict[cls]["recall"],
     "F1-Score": report_dict[cls]["f1-score"],
     "Support": report_dict[cls]["support"]}
    for cls in CLASSES
])

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(CLASSES))
w = 0.25

rects1 = ax.bar(x - w, perf_df["Precision"], w, label="Precision", color="#3498db", edgecolor="black", alpha=0.85)
rects2 = ax.bar(x, perf_df["Recall"], w, label="Recall", color="#2ecc71", edgecolor="black", alpha=0.85)
rects3 = ax.bar(x + w, perf_df["F1-Score"], w, label="F1-Score", color="#e74c3c", edgecolor="black", alpha=0.85)

ax.axhline(test_f1, color="red", linestyle="--", linewidth=1.5, label=f"Macro-F1 Avg: {test_f1:.3f}")
ax.set_title("Per-Class Precision, Recall, and F1-Score Breakdown", fontsize=14, fontweight="bold", pad=12)
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, fontsize=11)
ax.set_ylabel("Score (0.0 - 1.0)", fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(loc="upper right", frameon=True)
ax.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig(OUT / "plots" / "per_class_metrics.png", bbox_inches="tight", dpi=300)
plt.show()


---
## Step 17: Latent Space Embeddings & Class Similarity Analysis
Extract representations from the vision-language model, project to 2D via t-SNE / PCA, and compute class centroid cosine similarities.


In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

print("Projecting predictions and class probability distributions into 2D Latent Space...")

# Ensure test_probs is finite and sanitized (NaN/Inf protection)
test_probs = np.nan_to_num(test_probs, nan=1.0 / NUM_CLASSES, posinf=1.0, neginf=0.0)
test_probs = test_probs / np.clip(test_probs.sum(axis=1, keepdims=True), 1e-9, None)

# Perplexity bounded dynamically based on sample count
n_samples = len(test_probs)
perplexity = min(30, max(2, (n_samples - 1) // 4))

tsne = TSNE(n_components=2, random_state=CONFIG["seed"], perplexity=perplexity, init="pca" if n_samples > 10 else "random")
emb_2d = tsne.fit_transform(test_probs)
emb_2d = np.nan_to_num(emb_2d, nan=0.0, posinf=0.0, neginf=0.0)

# ----------------- Visual 10: 2D Latent Space Projection -----------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for cls_idx, cls_name in enumerate(CLASSES):
    mask = y_true == cls_idx
    if np.any(mask):
        axes[0].scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                        label=cls_name, color=CLASS_COLORS[cls_name], alpha=0.7, s=40, edgecolors="none")

axes[0].set_title("2D t-SNE Projection (Ground Truth Classes)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("t-SNE Dimension 1", fontsize=11)
axes[0].set_ylabel("t-SNE Dimension 2", fontsize=11)
axes[0].legend(frameon=True)
axes[0].grid(True, linestyle="--", alpha=0.4)

# Correct vs Misclassified in 2D space
axes[1].scatter(emb_2d[test_df["is_correct"], 0], emb_2d[test_df["is_correct"], 1],
                color="#2ecc71", alpha=0.6, s=35, label="Correctly Classified")
axes[1].scatter(emb_2d[~test_df["is_correct"], 0], emb_2d[~test_df["is_correct"], 1],
                color="#e74c3c", alpha=0.9, s=60, marker="x", label="Misclassified")
axes[1].set_title("Decision Boundary Quality (Correct vs Errors)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("t-SNE Dimension 1", fontsize=11)
axes[1].set_ylabel("t-SNE Dimension 2", fontsize=11)
axes[1].legend(frameon=True)
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(OUT / "plots" / "tsne_latent_space.png", bbox_inches="tight", dpi=300)
plt.show()

# ----------------- Visual 11: Class Centroid Cosine Similarity Heatmap -----------------
centroids = np.zeros((NUM_CLASSES, NUM_CLASSES))
for i in range(NUM_CLASSES):
    mask = y_true == i
    if np.any(mask):
        cls_probs = test_probs[mask]
        centroids[i] = np.nan_to_num(cls_probs.mean(axis=0), nan=1.0 / NUM_CLASSES)
    else:
        centroids[i, i] = 1.0

centroids = np.nan_to_num(centroids, nan=1.0 / NUM_CLASSES)
centroids = centroids / np.clip(np.linalg.norm(centroids, axis=1, keepdims=True), 1e-9, None)
sim_matrix = cosine_similarity(centroids)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sim_matrix, annot=True, fmt=".3f", cmap="YlGnBu",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_title("Class Centroid Cosine Similarity Matrix", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(OUT / "plots" / "class_centroid_similarity.png", bbox_inches="tight", dpi=300)
plt.show()


---
## Step 18: Misclassification & Confidence Calibration Analysis
Visualize confidence distributions for correct vs incorrect predictions, and inspect misclassified memes.


In [ ]:
# ----------------- Visual 12: Confidence Distribution -----------------
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

sns.histplot(data=test_df[test_df["is_correct"]], x="confidence", bins=25, color="#2ecc71", alpha=0.6, kde=True, ax=axes[0])
axes[0].set_title("Confidence Distribution: Correct Predictions", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Model Confidence (Softmax Probability)", fontsize=11)
axes[0].set_ylabel("Count", fontsize=11)

sns.histplot(data=test_df[~test_df["is_correct"]], x="confidence", bins=25, color="#e74c3c", alpha=0.6, kde=True, ax=axes[1])
axes[1].set_title("Confidence Distribution: Misclassifications", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Model Confidence (Softmax Probability)", fontsize=11)
axes[1].set_ylabel("Count", fontsize=11)

plt.tight_layout()
plt.savefig(OUT / "plots" / "confidence_calibration.png", bbox_inches="tight", dpi=300)
plt.show()

# ----------------- Visual 13: Misclassified Memes Gallery -----------------
errors_df = test_df[~test_df["is_correct"]].sort_values("confidence", ascending=False)
print(f"Total Test Misclassifications: {len(errors_df)} / {len(test_df)} ({len(errors_df)/len(test_df)*100:.1f}%)")

if len(errors_df) > 0:
    n_show = min(8, len(errors_df))
    cols = 4
    rows = math.ceil(n_show / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4.2 * rows))
    axes = axes.flatten() if rows > 1 else np.expand_dims(axes, axis=0).flatten()

    for i in range(len(axes)):
        if i < n_show:
            row = errors_df.iloc[i]
            try:
                img = Image.open(row["filepath"]).convert("RGB")
                axes[i].imshow(img)
                axes[i].set_title(f"True: {row['class']}\nPred: {row['predicted_class']} ({row['confidence']:.2f})",
                                  fontsize=10, fontweight="bold", color="#e74c3c")
            except Exception:
                axes[i].text(0.5, 0.5, "Image Load Error", ha="center")
        axes[i].axis("off")

    plt.tight_layout()
    plt.savefig(OUT / "plots" / "misclassified_examples.png", bbox_inches="tight", dpi=300)
    plt.show()


---
## Step 19: Single-Image Inference Demonstration
Test the fine-tuned model on an arbitrary single image path.


In [ ]:
def predict_single_image(image_path_or_record):
    if isinstance(image_path_or_record, str):
        path = image_path_or_record
        true_label = None
    else:
        path = image_path_or_record["filepath"]
        true_label = image_path_or_record.get("class", None)

    img = load_image(path, train=False)
    msgs = build_messages(img)
    prefix = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    proc = processor(text=[prefix], images=[img], return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in proc.items()}

    model.eval()
    with torch.no_grad():
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE, enabled=torch.cuda.is_available()):
            out = model(**inputs, use_cache=False)
        last_logits = out.logits[:, -1, :].float()
        scores = last_logits[:, FIRST_TOKEN_TENSOR]
        probs = torch.softmax(scores, dim=-1).cpu().numpy()[0]

    probs = np.nan_to_num(probs, nan=1.0 / NUM_CLASSES)
    pred_idx = probs.argmax()
    pred_cls = CLASSES[pred_idx]
    conf = probs[pred_idx]

    # Visual Display
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(12, 4))
    ax_img.imshow(img)
    title = f"Predicted: {pred_cls} ({conf*100:.1f}%)"
    if true_label:
        title += f"\nTrue Label: {true_label}"
    ax_img.set_title(title, fontsize=12, fontweight="bold", color="#27ae60" if true_label == pred_cls else "#e74c3c")
    ax_img.axis("off")

    bars = ax_bar.barh(CLASSES, probs * 100.0, color=PALETTE, edgecolor="black", alpha=0.85)
    ax_bar.set_xlim(0, 105)
    ax_bar.set_xlabel("Confidence (%)", fontsize=11)
    ax_bar.set_title("Class Probability Distribution", fontsize=12, fontweight="bold")
    ax_bar.grid(axis="x", linestyle="--", alpha=0.6)

    for bar in bars:
        w = bar.get_width()
        ax_bar.text(w + 1.5, bar.get_y() + bar.get_height()/2.0, f"{w:.1f}%", va="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()
    return pred_cls, conf

# Demonstrate on a test sample
if len(test_df) > 0:
    sample = test_df.iloc[0]
    predict_single_image(sample)


---
## Step 20: Summary & Exported Artifacts Overview


In [ ]:
print("=" * 60)
print("EXPERIMENT SUMMARY & EXPORTED ARTIFACTS")
print("=" * 60)

for p in sorted(OUT.rglob("*")):
    if p.is_file():
        rel = p.relative_to(OUT)
        size_kb = p.stat().st_size / 1024.0
        print(f" {str(rel):<40} ({size_kb:.1f} KB)")

print("\nNotebook execution finished successfully!")
